In [4]:
from typing import List, Dict, Any
import torch
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers.pipelines.base import Pipeline

In [5]:
# 初始化情緒分析 pipeline
classifier = pipeline("sentiment-analysis")
texts = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

# 執行預測
results: List[Dict[str, Any]] = classifier(texts)
print(f"預測結果: {results}\n")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 344.99it/s, Materializing param=pre_classifier.weight]                                  


預測結果: [{'label': 'POSITIVE', 'score': 0.9598049521446228}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]



In [6]:
checkpoint: str = "distilbert-base-uncased-finetuned-sst-2-english"

# 根據 checkpoint 載入對應的分詞器
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_inputs: List[str] = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
# 進行分詞處理，設定 padding 與 truncation，並指定輸出為 PyTorch
inputs = tokenizer(
        raw_inputs, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    )
print(f"處理後的輸入特徵 (Inputs): \n{inputs}\n")

處理後的輸入特徵 (Inputs): 
{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}



In [7]:
model = AutoModel.from_pretrained(checkpoint)
print(f"已成功載入基礎模型: {model.__class__.__name__}\n")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 346.92it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


已成功載入基礎模型: DistilBertModel



In [8]:
# 載入附帶序列分類頭的模型
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# 使用 ** 運算子解包字典，將 input_ids 與 attention_mask 傳入模型中進行推理
outputs = model(**inputs)

print(f"模型輸出 Logits: \n{outputs.logits}\n")

# [新增] 應用 Softmax 函數將 Logits 轉換為機率值 (Probabilities)
predictions: torch.Tensor = torch.nn.functional.softmax(outputs.logits, dim=-1)

print(f"預測機率值 (Probabilities): \n{predictions}\n")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 334.25it/s, Materializing param=pre_classifier.weight]                                  


模型輸出 Logits: 
tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)

預測機率值 (Probabilities): 
tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)

